In [4]:
!pip install shap lime scikit-learn torch xgboost lightgbm catboost scipy matplotlib seaborn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=7950e5c95f289b1e227eef241057d5b6496947a0fbf902c2453e54931a76ece2
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [5]:
import warnings
import os
import random
import copy
import itertools
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon, spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    ConfusionMatrixDisplay
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import shap
import lime
import lime.lime_tabular


In [6]:
SEED = 42

def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    os.environ["PYTHONHASHSEED"] = str(s)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# ## 1 . Data Loading

URL_MENDELEY = (
    "https://raw.githubusercontent.com/Md-Tanzim-Hossain/"
    "FederatedTransfer_Diabetes/refs/heads/main/"
    "Mendeley_Dataset%20of%20Diabetes.csv"
)
URL_PIMA = (
    "https://raw.githubusercontent.com/Md-Tanzim-Hossain/"
    "FederatedTransfer_Diabetes/refs/heads/main/PIMA_diabetes.csv"
)
URL_UCI = (
    "https://raw.githubusercontent.com/Md-Tanzim-Hossain/"
    "FederatedTransfer_Diabetes/refs/heads/main/"
    "UCI_diabetes_data_upload.csv"
)

def load_all_datasets():
    df_men = pd.read_csv(URL_MENDELEY)
    df_pim = pd.read_csv(URL_PIMA)
    df_uci = pd.read_csv(URL_UCI)

    for n, d in [("Mendeley", df_men), ("PIMA", df_pim), ("UCI", df_uci)]:
        print(f" {n}: {d.shape}")

    return df_men, df_pim, df_uci

print("[1] Loading datasets ...")
df_men_raw, df_pim_raw, df_uci_raw = load_all_datasets()

[1] Loading datasets ...
 Mendeley: (1000, 14)
 PIMA: (768, 9)
 UCI: (520, 17)


In [8]:
# ## 2 . Preprocessing

UCI_SYMPTOM_COLS = [
    "Polyuria", "Polydipsia", "sudden weight loss", "weakness", "Polyphagia",
    "Genital thrush", "visual blurring", "Itching", "Irritability", "delayed healing",
    "partial paresis", "muscle stiffness", "Alopecia", "Obesity"
]

UCI_FEATURE_COLS = ["Age", "Gender"] + UCI_SYMPTOM_COLS

MEN_FEATURE_COLS = ["Gender", "AGE", "Urea", "Cr", "HbA1c", "Chol", "TG", "HDL", "LDL", "VLDL", "BMI"]

PIMA_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

PIMA_FEATURE_COLS = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]

def preprocess_uci(df):
    df = df.copy()
    df["label"] = (df["class"].str.strip().str.lower() == "positive").astype(int)
    df["Gender"] = df["Gender"].str.strip().str.lower().map({"male": 1, "female": 0}).fillna(0).astype(int)

    for col in UCI_SYMPTOM_COLS:
        df[col] = df[col].str.strip().str.lower().map({"yes": 1, "no": 0}).fillna(0).astype(int)

    df["Age"] = pd.to_numeric(df["Age"], errors="coerce").fillna(df["Age"].median()).astype(float)
    df = df[UCI_FEATURE_COLS + ["label"]].reset_index(drop=True)

    print(f" UCI: {df.shape}, pos_rate={df['label'].mean():.3f}")
    return df, UCI_FEATURE_COLS[:]

def preprocess_mendeley(df):
    df = df.copy()
    df.drop(columns=["ID", "No_Pation"], inplace=True, errors="ignore")
    cls = df["CLASS"].str.strip().str.upper()
    df["label_3cls"] = cls.map({"N": 0, "P": 1, "Y": 2}).fillna(0).astype(int)
    df["label"] = (df["label_3cls"] == 2).astype(int)
    df["Gender"] = df["Gender"].str.strip().str.upper().map({"M": 1, "F": 0}).fillna(0).astype(int)

    for col in ["AGE", "Urea", "Cr", "HbA1c", "Chol", "TG", "HDL", "LDL", "VLDL", "BMI"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col].fillna(df[col].median(), inplace=True)

    df = df[MEN_FEATURE_COLS + ["label", "label_3cls"]].reset_index(drop=True)

    print(f" Mendeley: {df.shape}, pos_rate={df['label'].mean():.3f}")
    return df, MEN_FEATURE_COLS[:]

def preprocess_pima(df):
    df = df.copy()
    for col in PIMA_ZERO_COLS:
        df[col] = df[col].replace(0, np.nan)
        df[col].fillna(df[col].median(), inplace=True)

    df.rename(columns={"Outcome": "label"}, inplace=True)
    df = df[PIMA_FEATURE_COLS + ["label"]].reset_index(drop=True)

    print(f" PIMA: {df.shape}, pos_rate={df['label'].mean():.3f}")
    return df, PIMA_FEATURE_COLS[:]

df_uci, feats_uci = preprocess_uci(df_uci_raw)
df_men, feats_men = preprocess_mendeley(df_men_raw)
df_pim, feats_pim = preprocess_pima(df_pim_raw)


 UCI: (520, 17), pos_rate=0.615
 Mendeley: (1000, 13), pos_rate=0.844
 PIMA: (768, 9), pos_rate=0.349


In [ ]:
## Feature-Label Leakage Audit


def feature_label_audit(df, feature_cols, name):
    corrs = {}
    for col in feature_cols:
        r, _ = spearmanr(df[col], df["label"])
        corrs[col] = abs(r)

    corrs = dict(sorted(corrs.items(), key=lambda x: -x[1]))
    top5 = list(corrs.items())[:5]

    print(f"\n [{name}] Top-5 |Spearman rho| with label:")
    for feat, r in top5:
        print(f" {feat:35s}: rho={r:.4f}")

    max_r = max(corrs.values())
    n_high = sum(1 for v in corrs.values() if v > 0.7)
    print(f" Max rho={max_r:.4f}, features with rho>0.70: {n_high}")
    return corrs

print(" Feature-label audit ...")
corr_uci = feature_label_audit(df_uci, feats_uci, "UCI")
corr_men = feature_label_audit(df_men, feats_men, "Mendeley")
corr_pim = feature_label_audit(df_pim, feats_pim, "PIMA")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (corrs, name, col) in zip(axes, [
    (corr_uci, "UCI Sylhet", COLORS[0]),
    (corr_men, "Mendeley Iraqi", COLORS[1]),
    (corr_pim, "PIMA Indians", COLORS[2])
]):
    top = dict(list(corrs.items())[:10])
    ax.barh(list(top)[::-1], list(top.values())[::-1], color=col, alpha=0.8)
    ax.axvline(0.7, color="red", lw=1.2, ls="--", label="rho=0.70")
    ax.set(title=f"|Spearman rho| with label\n{name}", xlabel="|rho|", xlim=(0, 1))
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.3)

plt.suptitle("Feature-Label Correlation Audit", fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("feature_label_audit.png", dpi=600, bbox_inches="tight")
plt.show()

print("""
AUDIT INTERPRETATION (paper Methods/Limitations):
UCI Sylhet -- max rho=0.666. No single feature exceeds 0.70, but 14 binary symptom features create a near-linearly-separable space for tree ensembles. This reflects genuine clinical reality (co-occurrence of classic symptoms is strongly diagnostic) rather than data leakage. The federated contribution is privacy-preserving collaboration, not raw accuracy gain over local models.
Mendeley Iraqi -- max rho=0.573 (HbA1c). No leakage. Near-perfect baselines reflect HbA1c being the gold-standard diagnostic marker (>=6.5% per ADA).
PIMA Indians -- max rho=0.466 (Glucose). No single feature dominates; class imbalance (35% positive) makes this the most clinically meaningful benchmark. PIMA results are the primary evidence for federated utility in this study.
""")

 Feature-label audit ...

 [UCI] Top-5 |Spearman rho| with label:
 Polyuria                           : rho=0.6659
 Polydipsia                         : rho=0.6487
 Gender                             : rho=0.4492
 sudden weight loss                 : rho=0.4366
 partial paresis                    : rho=0.4323
 Max rho=0.6659, features with rho>0.70: 0

 [Mendeley] Top-5 |Spearman rho| with label:
 HbA1c                              : rho=0.5734
 BMI                                : rho=0.5679
 AGE                                : rho=0.4798
 TG                                 : rho=0.2245
 VLDL                               : rho=0.2245
 Max rho=0.5734, features with rho>0.70: 0

 [PIMA] Top-5 |Spearman rho| with label:
 Glucose                            : rho=0.4814
 Age                                : rho=0.3090
 BMI                                : rho=0.3070
 Insulin                            : rho=0.2744
 SkinThickness                      : rho=0.2157
 Max rho=0.4814, features

NameError: name 'COLORS' is not defined